In [ ]:
import os
import numpy as np

from tqdm import tqdm
from datasets import load_dataset, Audio
from dotenv import load_dotenv
from pathlib import Path

from audio_preprocessing import process_and_save
from config import DataConfig, AudioConfig

# Load HF Token, configs, and create cache dirs

In [ ]:
data_cfg = DataConfig()
audio_cfg = AudioConfig()

In [ ]:
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

In [ ]:
CACHE = Path(data_cfg.cache_dir)
CACHE.mkdir(parents=True, exist_ok=True)
(CACHE / "vctk").mkdir(exist_ok=True)
(CACHE / "nsynth").mkdir(exist_ok=True)

# Load and Process Data (Nsynth and VCTK)

In [ ]:
out_dir = CACHE / data_cfg.nsynth_subdir_name
budget = {"remaining_s": data_cfg.nsynth_length_hr * 3600.0}
print(f"NSynth budget: {budget['remaining_s'] / 3600:.1f}h")

ds = load_dataset(
    "parquet",
    data_files={
        "train": f"hf://datasets/{data_cfg.nsynth_hf_dataset_name}/data/train/*.parquet"
    },
    split="train",
    streaming=True,
    token=HF_TOKEN,
)
ds = ds.cast_column("audio", Audio(sampling_rate=audio_cfg.target_sr))
pbar = tqdm(total=int(budget["remaining_s"]), unit="s", desc="NSynth")

# NSynth notes are 4s. Filter for acoustic, mid-pitch (avoids extreme F0).
for i, ex in enumerate(ds):
    if budget["remaining_s"] <= 0:
        break
    pitch = ex.get("pitch", 60)
    # Notes C2..C6 — vocal-range overlap
    if pitch < 36 or pitch > 84:
        continue
    wav = ex["audio"]["array"].astype(np.float32)
    src_sr = ex["audio"]["sampling_rate"]
    sec = process_and_save(wav, src_sr, out_dir, f"n_{i:07d}", budget)
    pbar.update(int(sec))
pbar.close()

In [ ]:
out_dir = CACHE / data_cfg.vctk_subdir_name
budget = {"remaining_s": data_cfg.vctk_length_hr * 3600.0}
print(f"VCTK budget: {budget['remaining_s'] / 3600:.1f}h")

ds = load_dataset(
    "parquet",
    data_files={
        "train": f"hf://datasets/{data_cfg.vctk_hf_dataset_name}/data/train-*.parquet"
    },
    split="train",
    streaming=True,
    token=HF_TOKEN,
)
ds = ds.cast_column("audio", Audio(sampling_rate=audio_cfg.target_sr))
pbar = tqdm(total=int(budget["remaining_s"]), unit="s", desc="VCTK")

for i, ex in enumerate(ds):
    if budget["remaining_s"] <= 0:
        break
    wav = ex["audio"]["array"].astype(np.float32)
    src_sr = ex["audio"]["sampling_rate"]
    spk = ex.get("speaker_id", f"unk_{i}")
    sec = process_and_save(wav, src_sr, out_dir, f"v_{spk}_{i:06d}", budget)
    pbar.update(int(sec))
pbar.close()